In [3]:
import torch
import math
import random
from ase.io import read

In [4]:
pip install torchmd-net

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 MB 23.5 MB/s  0:00:05 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 30.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 45.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.3 MB/s  0:00:00
  Created wheel for torchmd-net: filename=torchmd_net-3.0.3-py3-none-any.whl size=172393 sha256=d84f50b55b8dc321f029f42e59084c1decb8c26cf96352c6e880b2e66af2dae4
  Stored in directory: /home/moises/.cache/pip/wheels/a7/91/6f/a253f281d921081120244161418fdc9543112

In [ ]:
import torch
import math
import random
from ase.io import read

# Importamos la red (asegúrate de tener spookynet instalada/accesible)
from spookynet import SpookyNet 

class SpookyBatch:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def __init__(self):
        self.N = 0
        self.Z = []
        self.R = []
        self.E = []
        self.F = [] # NUEVO: Lista para almacenar las fuerzas reales
        self.idx_i = []
        self.idx_j = []
        self.batch_seg = []

    def toTensor(self):
        self.Z = torch.tensor(self.Z, dtype=torch.int64, device=SpookyBatch.device)
        # R necesita requires_grad=True para poder derivar la energía y obtener fuerzas
        self.R = torch.tensor(self.R, dtype=torch.float32, device=SpookyBatch.device, requires_grad=True)
        self.Q = torch.zeros(self.N, dtype=torch.float32, device=SpookyBatch.device) 
        self.S = torch.zeros(self.N, dtype=torch.float32, device=SpookyBatch.device) 
        self.E = torch.tensor(self.E, dtype=torch.float32, device=SpookyBatch.device)
        self.F = torch.tensor(self.F, dtype=torch.float32, device=SpookyBatch.device) # NUEVO: Tensor de fuerzas
        self.idx_i = torch.tensor(self.idx_i, dtype=torch.int64, device=SpookyBatch.device)
        self.idx_j = torch.tensor(self.idx_j, dtype=torch.int64, device=SpookyBatch.device)
        self.batch_seg = torch.tensor(self.batch_seg, dtype=torch.int64, device=SpookyBatch.device) 
        return self

def load_batches_from_xyz(ruta_xyz, batch_size=8, stride=10):
    """
    Lee tu dinámica molecular de C240Na6 y la convierte en lotes de SpookyNet.
    stride=10 toma 1 de cada 10 frames para evitar datos repetitivos.
    """
    print(f"Cargando trayectoria desde {ruta_xyz}...")
    trayectoria = read(ruta_xyz, index=f"::{stride}")
    
    batches = []
    batch = None
    nm = 0 
    
    for m in trayectoria: 
        if nm == 0:
            na = 0 
            batch = SpookyBatch() 

        # Extraemos datos directamente del objeto de ASE
        numeros_atomicos = m.get_atomic_numbers().tolist()
        posiciones = m.get_positions().tolist()
        energia = m.get_potential_energy()
        fuerzas = m.get_forces().tolist()
        
        batch.Z.extend(numeros_atomicos)
        batch.R.extend(posiciones)
        batch.E.append(energia) 
        batch.F.extend(fuerzas) # Agregamos las fuerzas al batch
        
        cur_idx_i, cur_idx_j = get_idx(posiciones) 
        cur_idx_i += na
        cur_idx_j += na
        
        batch.idx_i.extend(cur_idx_i)
        batch.idx_j.extend(cur_idx_j)
        batch.batch_seg.extend([nm] * len(numeros_atomicos))
        
        na += len(numeros_atomicos)
        nm += 1

        if nm >= batch_size:
            batch.N = nm
            batches.append(batch.toTensor()) 
            nm = 0 
            
    if batch and nm > 0:
        batch.N = nm
        batches.append(batch.toTensor())
        
    print(f"Se crearon {len(batches)} lotes.")
    return batches

def get_idx(R):
    N = len(R)
    idx = torch.arange(N, dtype=torch.int64)
    idx_i = idx.view(-1, 1).expand(-1, N).reshape(-1)
    idx_j = idx.view(1, -1).expand(N, -1).reshape(-1)
    nidx_i = idx_i[idx_i != idx_j]
    nidx_j = idx_j[idx_i != idx_j]
    return nidx_i.numpy(), nidx_j.numpy() 

def train():
    NUM_EPOCHES = 1000
    BEST_POINT = 'spookynet_C240Na6_best.pt'
    START_LR = 1e-4
    RHO = 0.1 # NUEVO: 10% peso a energía, 90% a fuerzas (estándar en MD)

    model = SpookyNet().to(torch.float32).to(SpookyBatch.device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=START_LR, amsgrad=True)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=25, threshold=0)

    # 1. CARGAMOS TUS DATOS REALES (Asumiendo que dividiste tu .xyz en train y val)
    # Si tienes un solo archivo, deberías dividir la lista 'trayectoria' antes de hacer los batches
    training = load_batches_from_xyz("C240Na6_train.xyz", batch_size=4)
    validation = load_batches_from_xyz("C240Na6_val.xyz", batch_size=4)
    
    mse_sum = torch.nn.MSELoss(reduction='sum')
    mse_mean = torch.nn.MSELoss(reduction='mean') # Mejor para promediar el error de las fuerzas

    for epoch in range(NUM_EPOCHES):
        model.train()
        random.shuffle(training)
        loss_epoch = 0

        for batch in training:
            N = batch.N
            
            # Forward pass: predecir energía
            res = model.energy(Z=batch.Z, Q=batch.Q, S=batch.S, R=batch.R, 
                               idx_i=batch.idx_i, idx_j=batch.idx_j, 
                               batch_seg=batch.batch_seg, num_batch=N)
            E_pred = res[0]
            
            # NUEVO: Backward Pass Físico (Derivada de E respecto a R para obtener Fuerzas)
            F_pred = -torch.autograd.grad(
                outputs=E_pred, 
                inputs=batch.R, 
                grad_outputs=torch.ones_like(E_pred),
                create_graph=True, 
                retain_graph=True
            )[0]
     
            # Calculamos pérdida de Energía y Fuerzas
            loss_E = mse_sum(E_pred, batch.E) / N
            loss_F = mse_mean(F_pred, batch.F)
            
            # Pérdida combinada
            loss = (RHO * loss_E) + ((1.0 - RHO) * loss_F)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            loss_epoch += loss.item()

        # Evaluación en validación
        rmse_E, rmse_F = compute_rmse(validation, model)
        
        # Guardamos el mejor modelo basados en el error de fuerzas (crítico para MD)
        if scheduler.is_better(rmse_F, scheduler.best):
            torch.save(model.state_dict(), BEST_POINT)

        scheduler.step(rmse_F)
        print(f'Epoch: {epoch} / LR: {optimizer.param_groups[0]["lr"]:.6f} / RMSE Energía: {rmse_E:.4f} / RMSE Fuerzas: {rmse_F:.4f}')

def compute_rmse(batches, model):
    mse_sum = torch.nn.MSELoss(reduction='sum')
    total_mse_E = 0.0
    total_mse_F = 0.0
    count_E = 0
    count_F = 0
    
    model.eval()
    for batch in batches:
        N = batch.N
        
        # Necesitamos habilitar gradientes temporalmente incluso en validación para sacar las fuerzas
        with torch.enable_grad():
            res = model.energy(Z=batch.Z, Q=batch.Q, S=batch.S, R=batch.R, 
                               idx_i=batch.idx_i, idx_j=batch.idx_j, 
                               batch_seg=batch.batch_seg, num_batch=N)
            E_pred = res[0]
            
            F_pred = -torch.autograd.grad(
                outputs=E_pred, 
                inputs=batch.R, 
                grad_outputs=torch.ones_like(E_pred)
            )[0]

        total_mse_E += mse_sum(E_pred, batch.E).item()
        count_E += N
        
        # Para las fuerzas contamos todos los componentes (N_atomos * 3)
        total_mse_F += mse_sum(F_pred, batch.F).item()
        count_F += batch.F.numel() 

    model.train()
    return math.sqrt(total_mse_E / count_E), math.sqrt(total_mse_F / count_F)

if __name__ == "__main__":
    # Inicia el entrenamiento
    train()

# Training SpookyNet

In [1]:
import torch

class SpookyBatch:
    # Es mejor detectar automáticamente si hay una GPU disponible
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def __init__(self):
        self.num_frames = 0      # Cuántos pasos de tiempo (moléculas) hay en el batch
        self.num_atoms = 0       # Cuántos átomos totales hay apilados
        
        self.Z = []
        self.R = []
        self.E = []
        self.F = []              # [NUEVO] Fuerzas atómicas
        self.Q = []              # [CORREGIDO] Cargas
        self.S = []              # [CORREGIDO] Espines
        
        self.idx_i = []
        self.idx_j = []
        self.batch_seg = []

    def agregar_frame(self, z, r, e, f, q, s, idx_i, idx_j):
        """
        Esta función empaqueta un frame individual dentro de este batch gigante.
        """
        n_atoms_en_frame = len(z)
        
        # 1. Agregamos las propiedades básicas
        self.Z.extend(z)
        self.R.extend(r)
        self.E.append(e)
        self.F.extend(f)
        self.Q.append(q)
        self.S.append(s)
        
        # 2. Desplazamos los índices de los enlaces
        # (Para que el átomo 0 del frame 2 no se confunda con el átomo 0 del frame 1)
        self.idx_i.extend([i + self.num_atoms for i in idx_i])
        self.idx_j.extend([j + self.num_atoms for j in idx_j])
        
        # 3. Etiquetamos a qué frame pertenece cada átomo en este batch
        self.batch_seg.extend([self.num_frames] * n_atoms_en_frame)
        
        # Actualizamos contadores
        self.num_atoms += n_atoms_en_frame
        self.num_frames += 1

    def toTensor(self):
        """
        Convierte las listas acumuladas en tensores de PyTorch y los envía a la GPU.
        """
        # torch.long es el equivalente en PyTorch a int64
        self.Z = torch.tensor(self.Z, dtype=torch.long, device=SpookyBatch.device)
        
        # requires_grad=True es CRUCIAL en R para poder calcular las fuerzas con Autograd
        self.R = torch.tensor(self.R, dtype=torch.float32, device=SpookyBatch.device, requires_grad=True)
        
        self.E = torch.tensor(self.E, dtype=torch.float32, device=SpookyBatch.device)
        self.F = torch.tensor(self.F, dtype=torch.float32, device=SpookyBatch.device)
        self.Q = torch.tensor(self.Q, dtype=torch.float32, device=SpookyBatch.device)
        self.S = torch.tensor(self.S, dtype=torch.float32, device=SpookyBatch.device)
        
        self.idx_i = torch.tensor(self.idx_i, dtype=torch.long, device=SpookyBatch.device)
        self.idx_j = torch.tensor(self.idx_j, dtype=torch.long, device=SpookyBatch.device)
        self.batch_seg = torch.tensor(self.batch_seg, dtype=torch.long, device=SpookyBatch.device)
        
        return self

In [ ]:
def load_batches(my_mols): # my_mols == some structure which has your loaded mol data, prob retrieved from a file,
                                              # or you can load it from a file here on demand to save memory
    batches = []
    batch = None
    nm = 0 # how many mols in current batch
    NM = 100 # how many mols we put in each batch
    for  m in my_mols: 
        if nm == 0:
            na = 0 # num total atoms in this batch
            batch = SpookyBatch() # stores the data in a format we can pass to SpookyNet

         batch.Z.extend(m.species)
         batch.R.extend(m.coords)
         batch.E.append(m.energy) # target energy
         cur_idx_i,cur_idx_j = get_idx(m.coords) # see below but also look at SpookyNetCalculator for more options
         cur_idx_i += na
         cur_idx_j += na
         batch.idx_i.extend(cur_idx_i)
         batch.idx_j.extend(cur_idx_j)
         batch.batch_seg.extend([nm]*len(m.species))
         na += len(m.species)
         nm += 1

         if nm >= NM:
             batch.N = nm
             batches.append(batch.toTensor()) # or you could convert to a tensor during training, depends on how much memory you have
             nm = 0 
    if batch:
        batches.append(batch.toTensor()
    return batches

In [2]:
def get_idx(R, cutoff=5.0):
    """
    Calcula los índices de interacción basados en un radio de corte físico,
    ideal para evitar problemas de memoria en moléculas grandes.
    """
    # Nos aseguramos de que R sea un tensor para calcular distancias rápido
    if not isinstance(R, torch.Tensor):
        R = torch.tensor(R, dtype=torch.float32)
        
    # torch.cdist calcula una matriz con las distancias de todos contra todos
    dist_matrix = torch.cdist(R, R)
    
    # Creamos una máscara: distancia menor al cutoff y mayor a 0 
    # (dist > 1e-4 excluye las auto-interacciones porque la distancia de un átomo a sí mismo es 0)
    mask = (dist_matrix <= cutoff) & (dist_matrix > 1e-4)
    
    # torch.where nos da exactamente las coordenadas (i, j) donde la máscara es Verdadera
    idx_i, idx_j = torch.where(mask)
    
    # Lo regresamos como listas nativas de Python. 
    # Esto encaja perfecto con el '.extend()' de nuestro SpookyBatch.
    return idx_i.tolist(), idx_j.tolist()